# CSS Decoding Expanded Bivariate Bicycle Codes

This notebook loads bivariate bicycle CSS codes from `generator_matrices/bivariate_bicycle`, ensures expanded `Mqq` and `Mpp` lattice generators exist, and decodes one CSS sector using the same pattern as the surface-code CSS example.

In [ ]:
using LinearAlgebra
using LatticeDecoder
using NPZ
using Plots
using Random

Random.seed!(5);

Find the BB input directory and expand any missing files. The constructor writes `Mqq` and `Mpp` while preserving the original `hx`, `hz`, `lx`, and `lz` arrays.

In [ ]:
candidate_roots = unique([abspath(pwd()), abspath(joinpath(pwd(), ".."))])
root_index = findfirst(root -> isdir(joinpath(root, "generator_matrices", "bivariate_bicycle")), candidate_roots)
root_index === nothing && error("Could not find generator_matrices/bivariate_bicycle from $(pwd())")

project_root = candidate_roots[root_index]
input_dir = joinpath(project_root, "generator_matrices", "bivariate_bicycle")
expanded_dir = joinpath(input_dir, "expanded")
mkpath(expanded_dir)

function bb_prime_from_name(path::AbstractString)
    match_result = match(r"_p(\d+)(?:_expanded)?\.npz$", basename(path))
    match_result === nothing && return 2
    return parse(Int, match_result.captures[1])
end

bb_inputs = [
    file for file in sort(readdir(input_dir; join=true))
    if endswith(file, ".npz") && !occursin("_expanded.npz", basename(file))
]

constructor_rng = MersenneTwister(11)
constructor_max_iters = 10_000
balance_weights = true
rebuild_expanded = balance_weights

expanded_paths = String[]
for input_path in bb_inputs
    code_name = splitext(basename(input_path))[1]
    output_path = joinpath(expanded_dir, "$(code_name)_expanded.npz")

    output_has_generators = isfile(output_path) && all(k -> haskey(npzread(output_path), k), ("Mqq", "Mpp"))

    if rebuild_expanded || !output_has_generators
        loaded = load_sparse_quantum_code(input_path; hx_key=:hx, hz_key=:hz, balance_weights=balance_weights)
        enlarged = enlarge_css_generators(
            loaded.Hx,
            loaded.Hz;
            p=bb_prime_from_name(input_path),
            method=:heuristic,
            max_iters=constructor_max_iters,
            rng=constructor_rng,
        )

        output = Dict{String,Any}(loaded.data)
        output["Mqq"] = enlarged.Mqq
        output["Mpp"] = enlarged.Mpp
        npzwrite(output_path, output)
    end

    push!(expanded_paths, output_path)
end

basename.(expanded_paths)

Select one expanded code and one CSS sector. For `basis = "X"`, we decode with `Mqq` and use the dual `Mpp` generator for corrections. The stored BB matrices are integer generators, so we scale by `sqrt(p)` before decoding.

In [ ]:
function bb_css_sector(expanded_path; basis = "X")
    code = npzread(expanded_path)
    p = bb_prime_from_name(expanded_path)
    scale = sqrt(p)

    Mqq = Float64.(code["Mqq"]) / scale
    Mpp = Float64.(code["Mpp"]) / scale

    if basis == "X"
        logicals = Float64.(code["lx"]) / scale
        return Mqq, inv(Mpp), logicals, code
    elseif basis == "Z"
        logicals = Float64.(code["lz"]) / scale
        return Mpp, -inv(Mqq), logicals, code
    else
        error("basis must be either X or Z")
    end
end

code_index = 1
basis = "X"

expanded_path = expanded_paths[code_index]
H, G, logicals, code = bb_css_sector(expanded_path; basis = basis);
logical_check = inv(H);


Decode one Gaussian displacement with belief propagation. Local search is included as an opt-in follow-up because it can be expensive for these denser BB generator inverses.

In [ ]:
noise_std = 0.45
max_iter = size(H, 2)
decoder = "lsd"
search_interval = 1.0

error_vector = sample_error(noise_std, size(H, 2));
received = copy(error_vector);

tanner_graph = initialize_tanner_graph(H);
ldlc_decoder = LDLCDecoder(
    tanner_graph;
    schedule = :serial,
    algorithm = decoder,
    sigma = noise_std,
    max_iterations = max_iter,
    search_interval = search_interval,
)
bp_estimate = run_decoder!(ldlc_decoder, received);

decoded_integer_correction = hard_decision(bp_estimate, H);

run_local_search = false
local_search_integer_correction = copy(decoded_integer_correction);

if run_local_search
    order = [2, 1, 1];
    local_search_decoder = LocalSearch(length(order), G, order, false, false, false);
    λ = abs.(H * bp_estimate) .% 1.0;
    local_search!(received, λ, local_search_integer_correction, local_search_decoder);
end

Check whether the residual displacement is trivial up to the logical lattice.

In [ ]:
function is_not_logical_error(logical_check, residual; atol = 1e-5)
    logical_coordinates = logical_check' * residual
    return all(abs(x - round(x)) < atol for x in logical_coordinates)
end

bp_correction = received - G * decoded_integer_correction;
bp_residual = error_vector - bp_correction;

local_search_correction = received - G * local_search_integer_correction;
local_search_residual = error_vector - local_search_correction;

println("BP is correct: ", is_not_logical_error(logical_check, bp_residual))
if run_local_search
    println("BP + local search is correct: ", is_not_logical_error(logical_check, local_search_residual))
end
println("BP + logical is correct: ", is_not_logical_error(logical_check, bp_residual + logicals[1, :]))

## Tiny Smoke Sweep

This small sweep is intentionally tiny so the notebook stays interactive. Increase `samples_per_point` for smoother curves.

In [ ]:
function bb_decode_fails(H, G, noise_std;
    decoder = "nearest",
    search_interval = 1.0,
    local_search = false,
    local_search_decoder = nothing,
)
    logical_check = inv(H)
    error_vector = sample_error(noise_std, size(H, 2))
    received = copy(error_vector)
    tanner_graph = initialize_tanner_graph(H)

    ldlc_decoder = LDLCDecoder(
        tanner_graph;
        schedule = :serial,
        algorithm = decoder,
        sigma = noise_std,
        max_iterations = size(H, 2),
        search_interval = search_interval,
    )
    bp_estimate = run_decoder!(ldlc_decoder, received)
    decoded_integer_correction = hard_decision(bp_estimate, H)

    if local_search
        decoder_to_use = isnothing(local_search_decoder) ? LocalSearch(
            size(G, 2),
            G,
            ones(Int64, size(G, 2)),
            false,
            false,
            false,
        ) : local_search_decoder
        λ = abs.(H * bp_estimate) .% 1.0
        local_search!(received, λ, decoded_integer_correction, decoder_to_use)
    end

    correction = received - G * decoded_integer_correction
    residual = error_vector - correction

    return !is_not_logical_error(logical_check, residual)
end

sigmas = collect(range(0.25, 0.55; length = 4))
samples_per_point = 20
failure_rates = Dict{String, Vector{Float64}}()

for path in expanded_paths
    H_code, G_code, _, _ = bb_css_sector(path; basis = basis)
    rates = Float64[]

    for sigma in sigmas
        failures = count(_ -> bb_decode_fails(H_code, G_code, sigma; decoder = "nearest"), 1:samples_per_point)
        push!(rates, failures / samples_per_point)
    end

    failure_rates[basename(path)] = rates
end

failure_rates

In [ ]:
p = plot(
    xlabel = "noise std σ",
    ylabel = "logical failure rate",
    title = "Expanded BB CSS decoding ($(basis) sector)",
    legend = :topleft,
    yscale = :log10,
)

for path in expanded_paths
    label = basename(path)
    plot!(p, sigmas, failure_rates[label]; marker = :circle, label = label)
end

p